# Week 3 — ResNet50 Baseline Training and Evaluation (E0)

**Project:** Targeted Training Data Cleaning for Pest Classification on IP102  
**Status:** Reconstructed, executable notebook based on the recorded Week 3 workflow. It is **not** the original Colab session or a newly executed run.

The baseline was trained on all 45,095 training images, with model selection on validation Macro-F1. Test data were used only for final evaluation. The recorded best checkpoint was from **epoch 9**. Do not rerun training unless you intentionally want a new run.

## 1. Environment and dataset

Run in Google Colab with a GPU. The original run used a Tesla T4. The original archive was `ip102_v1.1.tar`, with `images/`, `train.txt`, `val.txt`, and `test.txt` under `ip102_v1.1/`. Data downloads are not embedded in this notebook; provide the extracted dataset in the location below.

In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score, recall_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = Path("/content/IP102_extracted/ip102_v1.1")
IMAGE_DIR = DATA_DIR / "images"
assert DATA_DIR.exists(), "Extract the IP102 dataset to DATA_DIR before proceeding."


## 2. Dataset and preprocessing

Training: resize to 256×256, random 224×224 crop, random horizontal flip, ImageNet normalization. Evaluation: resize to 256×256, center 224×224 crop, same normalization.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class IP102Dataset(Dataset):
    def __init__(self, split, transform=None):
        self.df = pd.read_csv(
            DATA_DIR / f"{split}.txt", sep=r"\s+", header=None,
            names=["filename", "class_id"]
        )
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(IMAGE_DIR / row["filename"]) as img:
            image = img.convert("RGB")
        label = int(row["class_id"])
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = IP102Dataset("train", train_transform)
val_dataset = IP102Dataset("val", eval_transform)
test_dataset = IP102Dataset("test", eval_transform)
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=(device.type == "cuda"))
print("Train:", len(train_dataset), "Validation:", len(val_dataset),
      "Test:", len(test_dataset))
assert (len(train_dataset), len(val_dataset), len(test_dataset)) == (45095, 7508, 22619)


## 3. Model and training configuration

ResNet50 with ImageNet-1K V2 pretrained weights, fully fine-tuned for 102 classes. AdamW (learning rate 1e-4, weight decay 1e-4), cross-entropy, 10 epochs, batch size 32, seed 42. Checkpoint selected using **validation Macro-F1**.

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 102)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
EPOCHS = 10

from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive/IP102_Proje/results/E0")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint location:", SAVE_DIR / "best_model.pth")


## 4. Baseline training — **optional; do not run if using the saved checkpoint**

The following code reconstructs the training loop. Running it will train a **new** model and may overwrite `best_model.pth` and `training_history.csv`. For the original Week 3 results, **skip this cell** and proceed to checkpoint loading.

In [ ]:
def evaluate_validation(model, loader):
    model.eval()
    all_labels, all_predictions = [], []
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(outputs.argmax(dim=1).cpu().numpy())
    return (total_loss / total_samples,
            accuracy_score(all_labels, all_predictions),
            f1_score(all_labels, all_predictions, labels=list(range(102)),
                     average="macro", zero_division=0))

# Set True ONLY to deliberately retrain the baseline.
RUN_TRAINING = False
if RUN_TRAINING:
    history, best_macro_f1 = [], -1.0
    for epoch in range(EPOCHS):
        model.train()
        running_loss, total_samples = 0.0, 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)
        train_loss = running_loss / total_samples
        val_loss, val_acc, val_f1 = evaluate_validation(model, val_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_f1:.4f}")
        history.append(dict(epoch=epoch+1, train_loss=train_loss,
                            val_loss=val_loss, val_accuracy=val_acc,
                            val_macro_f1=val_f1))
        if val_f1 > best_macro_f1:
            best_macro_f1 = val_f1
            torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_macro_f1": val_f1, "seed": SEED},
                       SAVE_DIR / "best_model.pth")
        pd.DataFrame(history).to_csv(SAVE_DIR / "training_history.csv", index=False)


## 5. Recorded validation history (original run)

These are **recorded results** supplied from the original Colab run, not outputs of this reconstructed notebook. The best validation Macro-F1 occurred at epoch 9.

In [ ]:
recorded_history = pd.DataFrame([
    [1,1.9845,1.3534,0.6275,0.4762],
    [2,1.1547,1.1655,0.6665,0.5530],
    [3,0.8989,1.1171,0.6893,0.5840],
    [4,0.7442,1.0894,0.6998,0.6114],
    [5,0.6198,1.1111,0.6973,0.6103],
    [6,0.5299,1.1362,0.7036,0.6206],
    [7,0.4556,1.1604,0.7036,0.6255],
    [8,0.3985,1.1942,0.7126,0.6327],
    [9,0.3519,1.2366,0.7108,0.6343],
    [10,0.3164,1.2622,0.7076,0.6252],
], columns=["epoch","train_loss","val_loss","val_accuracy","val_macro_f1"])
display(recorded_history)


## 6. Load the best checkpoint

The checkpoint is stored in Google Drive and is intentionally excluded from GitHub. Loading it avoids retraining. Only load checkpoints from a trusted source.

In [ ]:
CHECKPOINT_PATH = SAVE_DIR / "best_model.pth"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Best epoch:", checkpoint["epoch"])
print("Validation Macro-F1:", checkpoint["val_macro_f1"])


## 7. Test evaluation: Accuracy, Macro-F1, Tail Recall, ECE

Tail classes are the 21 least represented classes in the **original training split**. ECE uses 15 equal-width bins of maximum softmax confidence. **Do not use test metrics to tune training or cleaning decisions.**

In [ ]:
TAIL_CLASSES = [72,80,61,63,98,81,36,35,65,17,85,12,75,60,53,55,89,43,96,90,79]

def calculate_ece(confidences, predictions, labels, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    correct = predictions == labels
    ece = 0.0
    for i in range(n_bins):
        lower, upper = edges[i], edges[i+1]
        mask = ((confidences >= lower) & (confidences <= upper)) if i == 0 else (
            (confidences > lower) & (confidences <= upper))
        if not mask.any():
            continue
        ece += mask.mean() * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)

labels_all, preds_all, confs_all = [], [], []
model.eval()
with torch.inference_mode():
    for images, labels in test_loader:
        outputs = model(images.to(device, non_blocking=True))
        confidence, prediction = torch.softmax(outputs, dim=1).max(dim=1)
        labels_all.extend(labels.numpy())
        preds_all.extend(prediction.cpu().numpy())
        confs_all.extend(confidence.cpu().numpy())
labels_all = np.asarray(labels_all)
preds_all = np.asarray(preds_all)
confs_all = np.asarray(confs_all)

class_recalls = recall_score(labels_all, preds_all, labels=list(range(102)),
                             average=None, zero_division=0)
metrics = {
    "experiment": "E0",
    "best_epoch": int(checkpoint["epoch"]),
    "test_accuracy": float(accuracy_score(labels_all, preds_all)),
    "test_macro_f1": float(f1_score(labels_all, preds_all,
                                     labels=list(range(102)), average="macro", zero_division=0)),
    "tail_recall": float(np.mean(class_recalls[TAIL_CLASSES])),
    "ece_15_bins": calculate_ece(confs_all, preds_all, labels_all),
}
print(metrics)
pd.DataFrame([metrics]).to_csv(SAVE_DIR / "test_results.csv", index=False)
pd.DataFrame({"class_id": np.arange(102), "recall": class_recalls,
              "is_tail_class": [i in TAIL_CLASSES for i in range(102)]}).to_csv(
                  SAVE_DIR / "class_recalls.csv", index=False)
print("Results saved to:", SAVE_DIR)


## 8. Recorded E0 test results and interpretation

The **original** test run reported:

| Metric | Value |
|---|---:|
| Accuracy | 0.7081 |
| Macro-F1 | 0.6335 |
| Tail-class recall | 0.5303 |
| ECE (15 bins) | 0.1197 |

Validation Macro-F1 at the selected checkpoint: **0.6342780425519657** (epoch 9).

The tail-class recall is lower than overall accuracy, but these are different metrics and should not be interpreted as a direct like-for-like gap. Training loss decreased throughout training while validation loss increased after epoch 4, suggesting overfitting signs. This is a single-seed baseline, not an estimate of run-to-run variability.

### Reproducibility note
This notebook was reconstructed from the recorded code and logs. The original model checkpoint and CSV outputs reside in the project Google Drive, not in this repository. For an exact replication, also record package versions, the dataset archive version, and the scoring rules for subsequent experiments.